# TraceGuard — train on Colab, bring the weights home

Trains the frozen-backbone TraceGuard detector on a free Colab GPU and writes the result to
your Google Drive, so a dropped session never costs you a run.

**Before you run anything:** `Runtime → Change runtime type → T4 GPU`. Without this you get a
CPU box and training takes hours instead of minutes.

Run the cells top to bottom. Total time is roughly 30–45 minutes.

## 1. Confirm you actually got a GPU

If this errors or prints nothing, you are on a CPU runtime — go back and change it.

In [ ]:
!nvidia-smi

## 2. Get the code

**Set `BRANCH` below to whatever branch your work is on.** If your changes live on a branch and
this says `master`, Colab will cheerfully clone the old code and train the wrong model — with no
error message to warn you. The clone cell prints the commit it landed on and checks that the new
`--freeze-backbone` flag exists, so you can confirm you got the right code.

`GIT_LFS_SKIP_SMUDGE=1` skips downloading the old checkpoints, which you do not need here.

In [ ]:
%%bash -s "$REPO" "$BRANCH"
cd /content
rm -rf techjam
GIT_LFS_SKIP_SMUDGE=1 git clone --branch "$2" --single-branch "$1" techjam
cd techjam
pip install -q -e ".[dev]"
echo "--- installed, running on: ---"
git log --oneline -1
python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"
python -c "from traceguard.train import build_parser; print('freeze-backbone flag present:', '--freeze-backbone' in build_parser().format_help())"

In [ ]:
%%bash
cd /content
rm -rf techjam
GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/Jyang1206/techjam.git
cd techjam
pip install -q -e ".[dev]"
echo "--- installed ---"
python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"

## 3. Mount Google Drive

**This is the step that saves you from losing work.** Colab wipes `/content` when the session
ends, so everything gets written to Drive instead. The best checkpoint is re-saved every time
validation improves, so even a mid-run disconnect leaves you with the best model so far.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/traceguard/clip_run_001"
print("results will be written to:", OUTPUT_DIR)

## 4. Hugging Face login

Training streams SID_Set from Hugging Face. Without a token the connection is rate-limited and
can be dramatically slower. A free **Read** token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) is enough.

Paste it into the prompt below — don't hardcode it into the notebook, or it ends up on GitHub.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Train

What the important flags do:

| Flag | Why |
|---|---|
| `--backbone vit_large_patch14_clip_224.openai` | CLIP ViT-L/14, the backbone from Ojha et al. (CVPR 2023) |
| `--freeze-backbone` | Train only the small head. The backbone keeps its general visual knowledge instead of overwriting it with generator-specific trivia |
| `--degradation-probability 1.0` | Damage **every** training image. Top NTIRE 2026 entries used this as a strong regularizer |
| `--output-dir` on Drive | Survives a disconnect |

Roughly 4,800 trainable parameters out of 303 million — everything else is frozen.

If you hit an out-of-memory error, drop `--batch-size` to 32 or 16.

In [ ]:
%%bash -s "$OUTPUT_DIR"
cd /content/techjam
traceguard-train \
  --hf-dataset saberzl/SID_Set \
  --backbone vit_large_patch14_clip_224.openai \
  --freeze-backbone \
  --degradation-probability 1.0 \
  --max-train-samples 20000 \
  --max-validation-samples 4000 \
  --epochs 8 \
  --batch-size 64 \
  --workers 2 \
  --lr 1e-3 \
  --output-dir "$1"

## 6. Read the result

The number that matters is **validation ROC-AUC**. For reference, the EfficientNet baseline you
are replacing reached 0.7062 on held-out generators.

Also watch the two loss columns. If `train_loss` falls while `validation_loss` climbs, the model
is memorizing rather than learning — that is exactly the failure the frozen backbone is meant to
prevent, so it should look much flatter than the old run.

In [ ]:
import json

with open(f"{OUTPUT_DIR}/history.json") as handle:
    history = json.load(handle)

print(f"{'epoch':>5} {'train_loss':>11} {'val_loss':>9} {'val_auc':>8} {'bal_acc':>8}")
for row in history:
    print(
        f"{row['epoch']:>5} {row['train_loss']:>11.4f} {row['validation_loss']:>9.4f} "
        f"{row['roc_auc']:>8.4f} {row['balanced_accuracy']:>8.4f}"
    )

best = max(history, key=lambda row: row["roc_auc"])
print(f"\nbest epoch {best['epoch']}: ROC-AUC {best['roc_auc']:.4f}")
print("baseline to beat (EfficientNet, fine-tuned): 0.7062")

## 7. Optional — the ablation run

Same recipe with the frequency branch switched off. Comparing the two tells you whether that
branch earns its place, which is worth more to judges than an unmeasured architecture diagram.

Skip this if you are short on time.

In [ ]:
%%bash
cd /content/techjam
traceguard-train \
  --hf-dataset saberzl/SID_Set \
  --backbone vit_large_patch14_clip_224.openai \
  --freeze-backbone \
  --no-frequency \
  --degradation-probability 1.0 \
  --max-train-samples 20000 \
  --max-validation-samples 4000 \
  --epochs 8 \
  --batch-size 64 \
  --workers 2 \
  --lr 1e-3 \
  --output-dir /content/drive/MyDrive/traceguard/clip_run_002_nofreq

## 8. Bring the weights home

Two options:

**Easiest** — the files are already in your Drive at `MyDrive/traceguard/clip_run_001/`. Open
Drive in a browser and download `best.pt` and `history.json`.

**Direct** — run the cell below to download straight from Colab. `best.pt` is around 1.2 GB for
ViT-L/14, so browser download can be slow; Drive is usually less painful.

Once `best.pt` is on your laptop, put it somewhere like `checkpoints/clip/run_001/best.pt` and
everything local works unchanged:

```powershell
traceguard-demo --checkpoint checkpoints/clip/run_001/best.pt
traceguard-predict some_image_folder --checkpoint checkpoints/clip/run_001/best.pt
```

In [ ]:
from google.colab import files

files.download(f"{OUTPUT_DIR}/history.json")
files.download(f"{OUTPUT_DIR}/best.pt")